In [3]:
pip install pandas yfinance numpy matplotlib scipy streamlit


Note: you may need to restart the kernel to use updated packages.


In [4]:
streamlit run app.py

SyntaxError: invalid syntax (3737097518.py, line 1)

In [6]:
import streamlit as st
import pandas as pd
import yfinance as yf
import numpy as np
import matplotlib.pyplot as plt
from scipy.optimize import minimize

st.set_page_config(layout="wide")

# =========================
# SIDEBAR
# =========================
st.sidebar.title("Portfolio Settings")

uploaded_file = st.sidebar.file_uploader("Upload Portfolio CSV", type=["csv"])
manual_input = st.sidebar.text_input("Or enter tickers (comma separated)", "")

period = st.sidebar.selectbox(
    "Time Period",
    ["YTD", "1Y", "5Y", "10Y"]
)

benchmark = st.sidebar.text_input("Benchmark", value="SPY")

if st.sidebar.button("Refresh"):
    st.cache_data.clear()

# =========================
# DATE HANDLING
# =========================
period_map = {
    "YTD": "2026-01-01",
    "1Y": "2025-01-01",
    "5Y": "2021-01-01",
    "10Y": "2016-01-01"
}

start_date = period_map[period]

# =========================
# LOAD PORTFOLIO
# =========================
use_quantities = False

if uploaded_file:
    df = pd.read_csv(uploaded_file)
    tickers = df["ticker"].tolist()
    quantities = df["quantity"].values
    use_quantities = True

elif manual_input:
    tickers = [t.strip().upper() for t in manual_input.split(",")]
    weights = np.array([1/len(tickers)] * len(tickers))

else:
    st.warning("Using default portfolio: AAPL (30%), MSFT (40%), GOOGL (30%)")
    tickers = ["AAPL", "MSFT", "GOOGL"]
    weights = np.array([0.3, 0.4, 0.3])

# =========================
# DATA
# =========================
data = yf.download(tickers, start=start_date, auto_adjust=True)["Close"]
data = data.dropna()

returns = data.pct_change().dropna()

# Compute weights if quantities used
if use_quantities:
    latest_prices = data.iloc[-1]
    values = latest_prices * quantities
    weights = values / values.sum()

portfolio_returns = returns.dot(weights)
cumulative = (1 + portfolio_returns).cumprod()

# =========================
# BENCHMARK
# =========================
bench = yf.download(benchmark, start=start_date, auto_adjust=True)["Close"]
bench_returns = bench.pct_change().dropna()
bench_cum = (1 + bench_returns).cumprod()

# =========================
# RISK-FREE RATE
# =========================
rf_data = yf.download("^IRX", start=start_date)["Close"]
rf_rate = (rf_data / 100 / 252).squeeze()
rf_rate = rf_rate.reindex(portfolio_returns.index).ffill().bfill()

# =========================
# METRICS
# =========================
excess_returns = portfolio_returns - rf_rate

annual_return = float((1 + portfolio_returns.mean())**252 - 1)
volatility = float(portfolio_returns.std() * np.sqrt(252))
sharpe = float((excess_returns.mean() / excess_returns.std()) * np.sqrt(252))

# Align data
common_index = portfolio_returns.index.intersection(bench_returns.index)
p = portfolio_returns.loc[common_index].squeeze()
m = bench_returns.loc[common_index].squeeze()

# Beta / Alpha
beta = float(p.cov(m) / m.var())
rf_annual = float(rf_rate.mean() * 252)
market_return = float(m.mean() * 252)
capm = rf_annual + beta * (market_return - rf_annual)
alpha = float(annual_return - capm)

# Information Ratio
active_returns = p - m
tracking_error = active_returns.std() * np.sqrt(252)
ir = float((active_returns.mean() * 252) / tracking_error)

# =========================
# OPTIMIZATION
# =========================
def neg_sharpe(w):
    ret = np.sum(returns.mean() * w) * 252
    vol = np.sqrt(np.dot(w.T, np.dot(returns.cov() * 252, w)))
    return -(ret - rf_annual) / vol

def neg_ir(w):
    port = returns.dot(w)
    bench_local = bench_returns.loc[returns.index].squeeze()
    active = port - bench_local
    te = float(active.std() * np.sqrt(252))
    if np.isclose(te, 0):
        return 0
    return -(active.mean()*252)/te

constraints = {'type': 'eq', 'fun': lambda x: np.sum(x) - 1}
bounds = tuple((0, 1) for _ in tickers)
init = np.array([1/len(tickers)] * len(tickers))

opt_sharpe = minimize(neg_sharpe, init, method='SLSQP', bounds=bounds, constraints=constraints)
opt_ir = minimize(neg_ir, init, method='SLSQP', bounds=bounds, constraints=constraints)

w_sharpe = opt_sharpe.x
w_ir = opt_ir.x

opt_ret_sharpe = returns.dot(w_sharpe)
opt_ret_ir = returns.dot(w_ir)

opt_cum_sharpe = (1 + opt_ret_sharpe).cumprod()
opt_cum_ir = (1 + opt_ret_ir).cumprod()

# =========================
# DASHBOARD
# =========================
st.title("Portfolio Analytics Dashboard")

col1, col2 = st.columns(2)

with col1:
    fig1, ax = plt.subplots()
    ax.plot((cumulative - 1)*100, label="Portfolio")
    ax.plot((bench_cum - 1)*100, label="Benchmark")
    ax.plot((opt_cum_sharpe - 1)*100, label="Max Sharpe")
    ax.plot((opt_cum_ir - 1)*100, label="Max IR")
    ax.set_title("Performance Comparison")
    ax.legend()
    st.pyplot(fig1)

with col2:
    st.subheader("Metrics")
    st.write(f"Return: {annual_return:.2%}")
    st.write(f"Volatility: {volatility:.2%}")
    st.write(f"Sharpe: {sharpe:.2f}")
    st.write(f"Information Ratio: {ir:.2f}")
    st.write(f"Beta: {beta:.2f}")
    st.write(f"Alpha: {alpha:.2%}")

# =========================
# EFFICIENT FRONTIER
# =========================
st.subheader("Efficient Frontier")

n = 3000
results = np.zeros((3, n))

for i in range(n):
    w = np.random.random(len(tickers))
    w /= np.sum(w)

    ret = np.sum(returns.mean() * w) * 252
    vol = np.sqrt(np.dot(w.T, np.dot(returns.cov() * 252, w)))
    sr = (ret - rf_annual) / vol

    results[:, i] = [vol, ret, sr]

fig2, ax2 = plt.subplots()

scatter = ax2.scatter(results[0], results[1], c=results[2], cmap="viridis", alpha=0.5)
plt.colorbar(scatter, label="Sharpe")

ax2.scatter(volatility, annual_return, color="red", label="Original", s=100)
ax2.scatter(np.std(opt_ret_sharpe)*np.sqrt(252),
            np.mean(opt_ret_sharpe)*252,
            color="blue", label="Max Sharpe", s=100)

ax2.set_xlabel("Volatility")
ax2.set_ylabel("Return")
ax2.legend()

st.pyplot(fig2)

2026-03-26 20:53:59.237 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-03-26 20:53:59.239 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-03-26 20:53:59.239 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-03-26 20:53:59.240 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-03-26 20:53:59.241 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-03-26 20:53:59.242 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-03-26 20:53:59.243 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-03-26 20:53:59.243 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bar

DeltaGenerator()